# 🚀 Praca Magisterska - Główne Środowisko Colab
Ten notatnik to centrum dowodzenia eksperymentami do pracy magisterskiej.

**⚠️ Zanim zaczniesz:** Upewnij się, że włączone jest GPU w Google Colab: `Runtime (Środowisko wykonawcze)` -> `Change runtime type (Zmień typ środowiska)` -> `Hardware accelerator (Akcelerator sprzętowy)` -> wybierz `T4 GPU`.

In [ ]:
# ==========================================
# 1. INICJALIZACJA ŚRODOWISKA I POBRANIE KODU
# ==========================================
import os

# Przejście do katalogu głównego Colab
os.chdir('/content')

# Sklonowanie lub zaktualizowanie repozytorium
if not os.path.exists('Praca_magisterska'):
    !git clone https://github.com/Sornat11/Praca_magisterska.git
    os.chdir('Praca_magisterska')
else:
    os.chdir('Praca_magisterska')
    !git pull origin main

print("\n✅ Środowisko pomyślnie zsynchronizowane z najnowszą wersją GitHuba!")
!ls -la

In [ ]:
# ==========================================
# 2. INSTALACJA PAKIETÓW I APLIKOWANIE ŁATEK
# ==========================================
!pip install recbole hyperopt ray[tune] pandas pyyaml kmeans-pytorch matplotlib

# Załatana kompatybilność ze SciPy 1.11+ dla biblioteki RecBole (wymagane w nowym Colabie)
import recbole, os
path = os.path.join(os.path.dirname(recbole.__file__), 'model', 'general_recommender', 'lightgcn.py')
with open(path, 'r') as f:
    content = f.read()
content = content.replace('A._update(data_dict)', 'for k, v in data_dict.items(): A[k] = v')
with open(path, 'w') as f:
    f.write(content)
    
print("\n✅ Instalacja i łatki SciPy zaaplikowane!")

--- 
## 🧪 FAZA 0: Przygotowanie Danych (Uruchom tylko za pierwszym razem)
Pobiera i formatuje surowe dane MovieLens do formatu biblioteki RecBole.

In [ ]:
!python 1_Preprocessing/import_raw_data.py
!python 1_Preprocessing/convert_to_recbole.py

--- 
## 🧪 FAZA 1: Analiza Wrażliwości (Pojedyncze parametry) - INTERAKTYWNIE
Poniższy kod renderuje tabele i wykresy bezpośrednio tutaj w notatniku (w trybie wizualnym, a nie konsolowym).

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = '/content/Praca_magisterska'
sys.path.append(os.path.join(PROJECT_ROOT, '2_Experiments'))

from utils.recbole_patch import apply_patches
apply_patches()
from recbole.quick_start import run_recbole

# Funkcja rysująca wykres
def run_and_plot_colab(model_name, config_file_relative, param_name, param_values, dataset='ml-100k', metric='ndcg@10'):
    print(f"\n{'='*60}\n🚀 BADANIE PARAMETRU: {param_name} (Wartości: {param_values})\n{'='*60}")
    results = []
    config_file = os.path.join(PROJECT_ROOT, config_file_relative)
    
    for val in param_values:
        print(f" -> Trenowanie {model_name} z {param_name} = {val}...")
        config_dict = {
            param_name: val,
            'checkpoint_dir': os.path.join(PROJECT_ROOT, '3_Evaluation/Saved/'),
            'state': 'WARNING',
            'use_gpu': True,
            'gpu_id': '0',
            'device': 'cuda'
        }
        try:
            res_dict = run_recbole(model=model_name, dataset=dataset, config_file_list=[config_file],
                                   config_dict=config_dict, saved=False)
            score = res_dict.get('test_result', {}).get(metric, 0.0)
            results.append({param_name: val, metric: float(score)})
        except Exception as e:
            print(f" [BŁĄD] {e}")
            
    if not results: return
        
    df = pd.DataFrame(results)
    display(df)  # Interaktywna tabela
    
    plt.figure(figsize=(7, 4))
    plt.plot([str(v) for v in df[param_name]], df[metric], marker='o', linewidth=2.5, markersize=8, color='#ff7f0e')
    plt.title(f"Wpływ '{param_name}' na metrykę {metric}\n(Model: {model_name})", fontsize=12)
    plt.xlabel(param_name, fontsize=10)
    plt.ylabel(metric, fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.margins(0.1)
    plt.show()

In [ ]:
# Tutaj odpalasz badanie dla grafów (LightGCN)
run_and_plot_colab('LightGCN', '2_Experiments/Configs/lightgcn.yaml', 'n_layers', [1, 2, 3, 4, 5])

In [ ]:
# Możesz tu też dorzucić kolejne parametry, np. regularyzację
# run_and_plot_colab('LightGCN', '2_Experiments/Configs/lightgcn.yaml', 'reg_weight', [1e-1, 1e-2, 1e-3, 1e-4, 1e-5])

--- 
## 🧪 FAZA 2: Zautomatyzowane Strojenie Nocne (Grid Search / Bayes)
Uruchamia wielogodzinny trening na zadanej siatce. Najlepsze wartości automatycznie wypiszą się w konsoli po zakończeniu.

In [ ]:
# Optymalizacja NeuMF
#!python 2_Experiments/run_hyper.py --model NeuMF --config 2_Experiments/Configs/neumf.yaml --hyper 2_Experiments/Hyperparams/neumf.hyper --algo bayes

# Optymalizacja LightGCN
#!python 2_Experiments/run_hyper.py --model LightGCN --config 2_Experiments/Configs/lightgcn.yaml --hyper 2_Experiments/Hyperparams/gnn.hyper --algo bayes

# Optymalizacja BPR
#!python 2_Experiments/run_hyper.py --model BPR --config 2_Experiments/Configs/bpr.yaml --hyper 2_Experiments/Hyperparams/bpr.hyper --algo bayes

--- 
## 🧪 FAZA 3: Szybki Test Pojedynczego Modelu
Pozwala szybko uruchomić czysty model na z góry ustalonych, sztywnych hiperparametrach zdefiniowanych w pliku `.yaml`.

In [ ]:
#!python 2_Experiments/run_experiment.py --model BPR --dataset ml-100k --config 2_Experiments/Configs/bpr.yaml
#!python 2_Experiments/run_experiment.py --model LightGCN --dataset ml-100k --config 2_Experiments/Configs/lightgcn.yaml
#!python 2_Experiments/run_experiment.py --model NeuMF --dataset ml-100k --config 2_Experiments/Configs/neumf.yaml
#!python 2_Experiments/run_experiment.py --model ItemKNN --dataset ml-100k --config 2_Experiments/Configs/itemknn.yaml